In [1]:
#Name: Joshua Trabka
#Date: 06/05/2026
#Description: Migrates the data from the csv into a relational database



!pip install pandas
!pip install numpy
!pip install sqlalchemy
!pip install pymysql

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text, BigInteger, String
import pymysql

In [3]:
print("Step 1: Loading aac_shelter_outcomes.csv...")
df = pd.read_csv('aac_shelter_outcomes.csv')

# Drop arbitrary index if present
if '1' in df.columns:
    df = df.drop(columns=['1'])

# Handle empty values cleanly for SQL NULL conversion
df = df.replace({np.nan: None})

# Connection Settings
USER = 'root'
PASSWORD = 'Xy25!Jb4g86#'  
HOST = 'localhost'
PORT = '3306'
DB_NAME = 'aac_shelter_db'

engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}")

print("Step 2: Processing and deduplicating Dimension tables...")
dim_animals = df[['animal_id', 'name', 'date_of_birth', 'animal_type', 'breed', 'color']].copy()
dim_animals = dim_animals.dropna(subset=['animal_id'])
dim_animals = dim_animals.drop_duplicates(subset=['animal_id'])

dim_outcomes = df[['outcome_type', 'outcome_subtype']].copy().drop_duplicates()
dim_outcomes = dim_outcomes.dropna(subset=['outcome_type']).reset_index(drop=True)

try:
    # WORKSPACE CLEANUP
    print("Cleaning workspace and clearing old tables...")
    with engine.connect() as conn:
        conn.execute(text("DROP TABLE IF EXISTS Fact_Shelter_Outcomes;"))
        conn.execute(text("DROP TABLE IF EXISTS Dim_Animals;"))
        conn.execute(text("DROP TABLE IF EXISTS Dim_Outcomes;"))
        conn.commit()
        print(" -> Workspace wiped clean.")

    # INITIAL POPULATION WITH EXPLICIT TYPES
    print("Step 3: Exporting Dimension tables to MySQL via Pandas...")
    
    dim_animals.to_sql(
        'Dim_Animals', 
        con=engine, 
        if_exists='append', 
        index=False,
        dtype={'animal_id': String(50)}
    )
    
    dim_outcomes.to_sql(
        'Dim_Outcomes', 
        con=engine, 
        if_exists='append', 
        index=True, 
        index_label='outcome_id',
        dtype={'outcome_id': BigInteger}  
    )
    
    # Read back the exact index mappings from Dim_Outcomes
    db_outcomes = pd.read_sql(text('SELECT outcome_id, outcome_type, outcome_subtype FROM Dim_Outcomes'), con=engine)
    
    # Merge mapped IDs into central activity log
    merged_df = df.merge(db_outcomes, on=['outcome_type', 'outcome_subtype'], how='left')
    merged_df['datetime'] = pd.to_datetime(merged_df['datetime'], errors='coerce')

    # Data Sanitization
    merged_df = merged_df[merged_df['animal_id'].isin(dim_animals['animal_id'])]
    merged_df = merged_df[merged_df['outcome_id'].notna()]

    # Isolate final columns matching the Fact table parameters
    fact_outcomes = merged_df[[
        'animal_id', 'outcome_id', 'datetime', 'age_upon_outcome', 
        'age_upon_outcome_in_weeks', 'sex_upon_outcome', 'location_lat', 'location_long'
    ]].copy()

    print("Exporting Fact Table to MySQL...")
  
    fact_outcomes.to_sql(
        'Fact_Shelter_Outcomes', 
        con=engine, 
        if_exists='append', 
        index=False,
        dtype={
            'animal_id': String(50),
            'outcome_id': BigInteger  
        }
    )
    print(" -> Data successfully inserted into all tables.")

    # APPLY INTERCONNECTED RELATIONAL CONSTRAINTS
    print("Step 4: Altering database to enforce primary and foreign keys...")
    with engine.connect() as conn:
        # Define Primary Keys
        conn.execute(text("ALTER TABLE Dim_Animals ADD PRIMARY KEY (animal_id);"))
        conn.execute(text("ALTER TABLE Dim_Outcomes ADD PRIMARY KEY (outcome_id);"))
        
        # Add Auto-Increment Identifier to Fact Table records
        conn.execute(text("ALTER TABLE Fact_Shelter_Outcomes ADD COLUMN record_id BIGINT AUTO_INCREMENT PRIMARY KEY FIRST;"))
        
        # Bind everything together securely with Foreign Keys
        conn.execute(text("""
            ALTER TABLE Fact_Shelter_Outcomes 
            ADD CONSTRAINT fk_animal_link FOREIGN KEY (animal_id) REFERENCES Dim_Animals(animal_id) ON DELETE CASCADE,
            ADD CONSTRAINT fk_outcome_link FOREIGN KEY (outcome_id) REFERENCES Dim_Outcomes(outcome_id) ON DELETE RESTRICT;
        """))
        conn.commit()
        
    print("\n SUCCESS: Database creation and data migration completed with no errors!")

except Exception as e:
    print(f"\n ERROR DURING CONVERSION SYSTEM:\n{e}")

Step 1: Loading aac_shelter_outcomes.csv...
Step 2: Processing and deduplicating Dimension tables...
Cleaning workspace and clearing old tables...
 -> Workspace wiped clean.
Step 3: Exporting Dimension tables to MySQL via Pandas...


C:\Users\josht\AppData\Local\Temp\ipykernel_33548\4063696872.py:41: UserWarning: The provided table name 'Dim_Animals' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  dim_animals.to_sql(
C:\Users\josht\AppData\Local\Temp\ipykernel_33548\4063696872.py:49: UserWarning: The provided table name 'Dim_Outcomes' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  dim_outcomes.to_sql(


Exporting Fact Table to MySQL...


C:\Users\josht\AppData\Local\Temp\ipykernel_33548\4063696872.py:77: UserWarning: The provided table name 'Fact_Shelter_Outcomes' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  fact_outcomes.to_sql(


 -> Data successfully inserted into all tables.
Step 4: Altering database to enforce primary and foreign keys...

🎉 SUCCESS: Database creation and data migration completed with no errors!
